# DEDALUS · Perception · YOLO su un video di CARLA (Kaggle)
**YOLO trova veicoli e pedoni in un video registrato dallo schermo di CARLA.**
È il piano B per la demo: il PC di Alessandro non riesce a registrare i sensori, ma può registrare lo schermo.
Senza camera di profondità **non abbiamo la distanza**: quella la aggiungiamo quando avremo una macchina più potente.

### Prima di iniziare (una volta sola)
1. Account Kaggle con il **telefono verificato** (*Settings → Phone verification*).
2. A destra, in **Settings**: *Accelerator* → **GPU T4 x2** (o P100), *Internet* → **On**.
3. **Add Input → Upload** → carica il video (`.mp4`) come nuovo dataset, per esempio con il nome `video-carla`.

### Poi
Menu **Run All**. Per cambiare qualcosa, cambiala nella cella **2 · Impostazioni** e rilancia da lì in giù.

## 1 · Installazione e controllo della GPU

In [ ]:
!pip install -q ultralytics

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATTENZIONE: nessuna GPU. Settings -> Accelerator -> GPU T4 (serve il telefono verificato)")

## 2 · Impostazioni
La cella cerca da sola **tutti** i video `.mp4` dentro `/kaggle/input` e li analizza uno dopo l'altro (per esempio CLIP1, CLIP2, CLIP3).
Per usarne solo uno, scrivi a mano la lista in `VIDEOS`, per esempio `VIDEOS = [video_trovati[0]]`.

**Video lungo?** Non serve tagliarlo: scegliete il pezzo dove la strada si vede bene e scrivete l'inizio e la fine in `INIZIO_SECONDI` e `FINE_SECONDI` (valgono per ogni video). Con `FINE_SECONDI = 9999` si analizza tutto.

In [ ]:
import glob, os

BASE_INPUT = "/kaggle/input"
BASE_OUTPUT = "/kaggle/working"

video_trovati = sorted(glob.glob(os.path.join(BASE_INPUT, "**", "*.mp4"), recursive=True))
print("Video trovati:")
for v in video_trovati:
    print("  ", v)
if not video_trovati:
    raise SystemExit("Nessun video .mp4 trovato: hai aggiunto il dataset con Add Input?")

VIDEOS = video_trovati            # <-- tutti i video trovati (oppure una lista scritta a mano)
CARTELLA_RISULTATI = os.path.join(BASE_OUTPUT, "risultati_video")

MODELLO = "yolo26n.pt"            # n = nano (il piu' veloce). Prova anche yolo26s.pt
CONFIDENZA_MINIMA = 0.3           # sotto questo valore ignoriamo il riquadro
OGNI_QUANTI = 3                   # analizziamo 1 fotogramma ogni 3 (video a 30 al secondo -> 10 al secondo)
INIZIO_SECONDI = 0                # da che secondo del video partire (es. 95 = 1 minuto e 35)
FINE_SECONDI = 9999               # a che secondo fermarsi (9999 = fino alla fine del video)

CLASSI_UTILI = {"person": "pedone", "bicycle": "veicolo", "car": "veicolo",
                "motorcycle": "veicolo", "bus": "veicolo", "truck": "veicolo"}
COLORI = {"veicolo": (0, 165, 255), "pedone": (255, 0, 255)}   # arancione, magenta (B, G, R)

print("\nVideo da analizzare:", len(VIDEOS))

## 3 · YOLO sul video
Leggiamo ogni video un fotogramma alla volta (`cv2.VideoCapture`), come se fossero le foto di una sessione.
Per ogni fotogramma analizzato salviamo un'immagine con i riquadri (il nome comincia con il nome del video, per esempio `CLIP1_000042.jpg`) e lo aggiungiamo al video dei risultati di quel video (`CLIP1_yolo.mp4`).

In [ ]:
import time
import cv2
from ultralytics import YOLO

modello = YOLO(MODELLO)          # la prima volta scarica il modello da internet
os.makedirs(CARTELLA_RISULTATI, exist_ok=True)

tempi = []
righe = ["foto,secondo,classe,confidenza,larghezza_px,altezza_px"]
oggetti_per_foto = {}
nomi = []

for VIDEO in VIDEOS:
    clip = os.path.splitext(os.path.basename(VIDEO))[0]     # es. "CLIP1"
    lettore = cv2.VideoCapture(VIDEO)
    fps_video = lettore.get(cv2.CAP_PROP_FPS) or 30
    print("Analizzo", clip, "...")
    video_out = None
    numero = -1

    while True:
        ok, img = lettore.read()     # ok = False quando il video e' finito
        if not ok:
            break
        numero += 1
        if numero % OGNI_QUANTI != 0:
            continue                 # saltiamo questo fotogramma
        secondo = numero / fps_video
        if secondo < INIZIO_SECONDI:
            continue                 # non siamo ancora arrivati al pezzo scelto
        if secondo > FINE_SECONDI:
            break
        nome = f"{clip}_{numero:06d}"

        inizio = time.time()
        risultato = modello(img, conf=CONFIDENZA_MINIMA, verbose=False)[0]
        ms = (time.time() - inizio) * 1000
        tempi.append(ms)

        trovati = []
        for riquadro in risultato.boxes:
            nome_yolo = modello.names[int(riquadro.cls)]
            if nome_yolo not in CLASSI_UTILI:
                continue
            classe = CLASSI_UTILI[nome_yolo]
            confidenza = float(riquadro.conf)
            x1, y1, x2, y2 = [int(v) for v in riquadro.xyxy[0].tolist()]
            righe.append(f"{nome},{secondo:.1f},{classe},{confidenza:.2f},{x2 - x1},{y2 - y1}")
            trovati.append(classe)

            colore = COLORI[classe]
            cv2.rectangle(img, (x1, y1), (x2, y2), colore, 3)
            cv2.putText(img, f"{nome_yolo} {confidenza:.2f}", (x1, max(y1 - 8, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, colore, 2)

        oggetti_per_foto[nome] = trovati
        nomi.append(nome)
        cv2.putText(img, f"{clip}  {secondo:.1f} s  |  {ms:.0f} ms", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        cv2.imwrite(os.path.join(CARTELLA_RISULTATI, nome + ".jpg"), img)
        if video_out is None:
            video_out = cv2.VideoWriter(os.path.join(CARTELLA_RISULTATI, clip + "_yolo.mp4"),
                                        cv2.VideoWriter_fourcc(*"mp4v"), fps_video / OGNI_QUANTI,
                                        (img.shape[1], img.shape[0]))
        video_out.write(img)

    lettore.release()
    if video_out is None:
        print("  ATTENZIONE: nessun fotogramma letto da", clip)
    else:
        video_out.release()

if not nomi:
    raise SystemExit("Non sono riuscito a leggere nessun fotogramma: i video sono rovinati?")
with open(os.path.join(CARTELLA_RISULTATI, "rilevamenti.csv"), "w") as f:
    f.write("\n".join(righe))
print("Fatto! Fotogrammi analizzati:", len(nomi))

## 4 · Velocità (per la slide dei risultati)

In [ ]:
tempi_utili = tempi[1:] if len(tempi) > 1 else tempi     # il primo fotogramma e' lento (YOLO si "scalda")
media = sum(tempi_utili) / len(tempi_utili)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Tempo medio: {media:.1f} ms per immagine  ->  {1000 / media:.1f} immagini al secondo")
print(f"Hardware: Kaggle, {gpu}")
print()
print("Frase per la slide:")
print(f"  Speed: {media:.0f} ms per image ({1000 / media:.0f} FPS) on Kaggle {gpu}")

## 5 · Cosa ha trovato

In [ ]:
import pandas as pd
tabella = pd.read_csv(os.path.join(CARTELLA_RISULTATI, "rilevamenti.csv"), dtype={"foto": str})
print("Oggetti trovati in totale:", len(tabella))
if len(tabella) > 0:
    print(tabella["classe"].value_counts().to_string())
vuoti = sum(1 for k in oggetti_per_foto if len(oggetti_per_foto[k]) == 0)
print(f"Fotogrammi senza nessun oggetto trovato: {vuoti} su {len(nomi)}")
tabella.head(15)

## 6 · Immagini da guardare (per scegliere gli esempi)
- **A**: i fotogrammi con **più oggetti** trovati: candidati per "dove funziona".
- **B**: fotogrammi a caso: cercate cosa **manca** (auto non segnate, pedoni lontani, auto viste dall'alto…). YOLO non segna ciò che non trova!

In [ ]:
import matplotlib.pyplot as plt
import random

def mostra(nomi_scelti, titolo):
    n = len(nomi_scelti)
    if n == 0:
        print("niente da mostrare")
        return
    colonne = 2
    righe_fig = (n + colonne - 1) // colonne
    plt.figure(figsize=(16, 4.6 * righe_fig))
    for i, nome in enumerate(nomi_scelti):
        img = cv2.imread(os.path.join(CARTELLA_RISULTATI, nome + ".jpg"))
        plt.subplot(righe_fig, colonne, i + 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(nome)
        plt.axis("off")
    plt.suptitle(titolo, fontsize=16)
    plt.tight_layout()
    plt.show()

piu_oggetti = sorted(oggetti_per_foto, key=lambda k: len(oggetti_per_foto[k]), reverse=True)[:4]
mostra(piu_oggetti, "A · Fotogrammi con piu' oggetti trovati")

random.seed(0)
a_caso = random.sample(nomi, min(4, len(nomi)))
mostra(a_caso, "B · Fotogrammi a caso: cosa manca?")

## 7 · Scaricare i risultati
Dopo questa cella, a destra nel pannello **Output** trovi `risultati.zip`: dentro ci sono le immagini con i riquadri, un video con i riquadri per ogni clip (`CLIP1_yolo.mp4`…) e `rilevamenti.csv`.

In [ ]:
import shutil
archivio = shutil.make_archive(os.path.join(BASE_OUTPUT, "risultati"), "zip", CARTELLA_RISULTATI)
print("Pronto da scaricare:", archivio)

## Per la slide dei risultati
1. Scegli **un'immagine dove funziona** e **una dove sbaglia**.
2. Sotto ciascuna, una frase: es. *"Cars found at street level"*, *"Cars seen from above are missed"*.
3. La frase della **cella 4** con millisecondi, FPS e GPU.
4. Ricorda di dire: *"No distance yet: it needs the depth camera, which we record as soon as we have a stronger machine."*